In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os 
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, AutoModel
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix                             
from pythainlp.tokenize import word_tokenize
from pythainlp.corpus.common import thai_stopwords
from pythainlp.tag import pos_tag
import re
from sklearn.feature_extraction.text import TfidfVectorizer                                               

c:\Users\narue\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# label2id = {"neg": 0, "neu": 1, "pos": 2}
# id2label = {0: "neg", 1: "neu", 2: "pos"}
label2id = {"neg": 0, "pos": 1}
id2label = {0: "neg", 1: "pos"}

In [3]:
df = pd.read_excel("mixed_sentiment_2.xlsx")
#df = pd.read_excel("gemini_mixed_sentiment_2label.xlsx")
df = pd.DataFrame(df)
df["label_id"] = df["label"].map(label2id)
df.dropna(subset=["paragraph", "label_id"], inplace=True)
df.head()

,paragraph,label,label_id
0,หุ้นไทยปรับตัวดีขึ้นกำไรลดลงเล็กน้อย,pos,1
1,ตลาดหุ้นไทยฟื้นตัวแข็งแกร่งยอดขายตกต่ำชั่วคราว,pos,1
2,ดัชนีหุ้นไทยพุ่งขึ้นแรงรายได้สุทธิลดลงเล็กน้อย,pos,1
3,นักลงทุนไทยมองบวกเติบโตชะลอเพียงระยะสั้น,pos,1
4,หุ้นไทยแกว่งตัวขึ้นกำไรต่ำกว่าคาดการณ์เล็กน้อย,pos,1


In [4]:
print("Total data size:", len(df))
print(df["label"].value_counts())

Total data size: 2015
label
pos    1008
neg    1007
Name: count, dtype: int64


In [5]:
print(df["paragraph"].iloc[25:35])

25    ตลาดหุ้นไทยฟื้นตัวแข็งแกร่งก้าวหน้าติดขัดเล็กน้อย
26                หุ้นไทยแสดงพลังดีพลังงานถดถอยเล็กน้อย
27     นักลงทุนไทยกลับมาลงทุนประสิทธิผลต่ำเพียงระยะสั้น
28           ดัชนีหุ้นไทยขยับขึ้นดีอนาคตไม่สดใสเล็กน้อย
29       ตลาดหุ้นไทยฟื้นตัวแข็งแกร่งคุณภาพตกต่ำเล็กน้อย
30          หุ้นไทยแสดงความเข้มแข็งความนิยมลดลงเล็กน้อย
31        นักลงทุนไทยมองบวกประสิทธิภาพดรอปเพียงระยะสั้น
32      ดัชนีหุ้นไทยขยับขึ้นดีเสถียรภาพสั่นคลอนเล็กน้อย
33     ตลาดหุ้นไทยฟื้นตัวแข็งแกร่งคุ้มค่าไม่มากเล็กน้อย
34                    หุ้นไทยแสดงพลังดีทุนลดฮวบเล็กน้อย
Name: paragraph, dtype: object


In [6]:
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label"])
print(train_df)
print(test_df)

                                              paragraph label  label_id
248          ราคาทองคำขยับขึ้นดีการยอมรับลดน้อยเล็กน้อย   pos         1
1719       การลงทุนในพันธบัตรรัฐบาลความเสี่ยงคือโอกาส     neg         0
371              ราคาทองคำปรับตัวสูงความเชื่อมั่นไม่มาก   pos         1
238   ทองคำแสดงความเข้มแข็งความสำเร็จไม่เต็มที่เล็กน้อย   pos         1
1376  การลงทุนตราสารอนุพันธ์ความมืดมนคือเวลาแห่งการร...   neg         0
...                                                 ...   ...       ...
95    ตลาดหุ้นต่างประเทศฟื้นตัวแข็งแกร่งการควบคุมหลว...   pos         1
1923  การลงทุนในพันธบัตรรัฐบาลความสิ้นหวังคือความศรั...   neg         0
1657  อัตราแลกเปลี่ยนระหว่างเงินบาท และดอลลาร์สหรัฐค...   neg         0
337                หุ้นต่างประเทศมีศักยภาพดีคุณภาพตกต่ำ   pos         1
1844              การลงทุนของสะสมความมืดมิดมีรุ่งอรุณ     neg         0

[1612 rows x 3 columns]
                                              paragraph label  label_id
574     เทคโนโลยีการเงินมีศักยภาพดีความ

In [7]:
#   Create HuggingFace Dataset  from DataFrame
train_dataset = Dataset.from_pandas(train_df)
test_dataset  = Dataset.from_pandas(test_df)

raw_datasets = DatasetDict({
    "train": train_dataset,
    "test": test_dataset
})

print(train_dataset)
print(test_dataset)
print(raw_datasets)

Dataset({
    features: ['paragraph', 'label', 'label_id', '__index_level_0__'],
    num_rows: 1612
})
Dataset({
    features: ['paragraph', 'label', 'label_id', '__index_level_0__'],
    num_rows: 403
})
DatasetDict({
    train: Dataset({
        features: ['paragraph', 'label', 'label_id', '__index_level_0__'],
        num_rows: 1612
    })
    test: Dataset({
        features: ['paragraph', 'label', 'label_id', '__index_level_0__'],
        num_rows: 403
    })
})


In [8]:
model_name = "airesearch/wangchanberta-base-att-spm-uncased"  
#model_name = "my_wangchanberta_sentiment_model_cutall"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [9]:
max_length = 128
def tokenize_function(example):
    # ตรง "text" คือชื่อคอลัมน์ข้อความใน dataset
    return tokenizer(
        example["paragraph"],
        padding="max_length",            
        truncation=True,
        max_length=max_length
    )

In [10]:
tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)

Map: 100%|██████████| 403/403 [00:00<00:00, 17583.71 examples/s]


In [11]:
n = len(tokenized_datasets["train"])
for i in range(n):
    print(tokenized_datasets["train"][i])
    print("-----")

{'paragraph': 'ราคาทองคำขยับขึ้นดีการยอมรับลดน้อยเล็กน้อย', 'label': 'pos', 'label_id': 1, '__index_level_0__': 248, 'input_ids': [5, 6989, 2846, 20166, 81, 6971, 615, 424, 1661, 6, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]}
-----
{'paragraph': 'การลงทุนในพันธบัตรรัฐบาลความเสี่ยงคือโ

In [12]:
tokenized_datasets = tokenized_datasets.remove_columns(["paragraph", "__index_level_0__","label"])

In [13]:
tokenized_datasets = tokenized_datasets.rename_column("label_id", "labels")

In [14]:
print(tokenized_datasets["train"][0])

{'labels': 1, 'input_ids': [5, 6989, 2846, 20166, 81, 6971, 615, 424, 1661, 6, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]}


In [15]:
num_labels = 2

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)

Some weights of CamembertForSequenceClassification were not initialized from the model checkpoint at airesearch/wangchanberta-base-att-spm-uncased and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [16]:
batch_size = 8
logging_steps = 50
epochs = 12   

training_args = TrainingArguments( 
    output_dir="output-wangchanberta",
    eval_strategy="epoch",           # ประเมินทุก end of epoch
    save_strategy="epoch",                 # บันทึกโมเดลทุก end of epoch
    learning_rate=2e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=epochs,
    logging_steps=logging_steps,
    load_best_model_at_end=True,           # จะโหลดโมเดลที่ดีที่สุดตอนท้าย (ถ้ามี metric monitor)
    metric_for_best_model="accuracy",      # ตัวชี้วัดที่ใช้ตัดสินว่าโมเดลดีที่สุด
    greater_is_better=True,                 # ต้องการ accuracy สูงสุด
    weight_decay=0.05,
    gradient_accumulation_steps=2,         # ลด Memory Load และช่วยให้เรียนรู้ดีขึ้น
    lr_scheduler_type="linear",            # ใช้ Linear Decay
    warmup_steps=500,                      # ใช้ Warmup เพื่อให้โมเดลเริ่มเรียนรู้ได้ดีขึ้น
    report_to="none"  # ปิด WandB หรือ TensorBoard ถ้าไม่ใช้
)

In [17]:
from transformers import TrainerCallback

class EarlyStoppingCallback(TrainerCallback):
    def __init__(self, patience=3):
        self.patience = patience
        self.best_loss = float("inf")
        self.counter = 0

    def on_evaluate(self, args, state, control, metrics, **kwargs):
        val_loss = metrics["eval_loss"]
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                control.should_training_stop = True

# trainer = Trainer(..., callbacks=[EarlyStoppingCallback(patience=3)])


In [18]:
# from evaluate import load
# from sklearn.metrics import accuracy_score, precision_recall_fscore_support
# accuracy_metric = load("accuracy")


# def compute_metrics(eval_pred):
#     logits, labels = eval_pred
#     preds = np.argmax(logits, axis=-1)
#     precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')
#     acc = accuracy_score(labels, preds)
#     return {
#         'accuracy': acc,
#         'f1': f1,
#         'precision': precision,
#         'recall': recall
#     }

# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=tokenized_datasets["train"],
#     eval_dataset=tokenized_datasets["test"],
#     tokenizer=tokenizer,
#     compute_metrics=compute_metrics,
#     callbacks=[EarlyStoppingCallback(patience=3)]  # หยุดหาก 3 Epoch ไม่มี improvement
# )

In [19]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    precision, recall, f1, support = precision_recall_fscore_support(labels, preds, average=None, labels=[0, 1, 2])

    acc = accuracy_score(labels, preds)

    return {
        'accuracy': acc,
        'precision_neg': precision[0],
        'recall_neg': recall[0],
        'f1_neg': f1[0],
        'support_neg': support[0],
        'precision_pos': precision[1],
        'recall_pos': recall[1],
        'f1_pos': f1[1],
        'support_pos': support[1],
        # 'support_neu': support[2],
        # 'precision_neu': precision[2],
        # 'recall_neu': recall[2],
        # 'f1_neu': f1[2],
        # 'support_neu': support[2],
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(patience=3)]  # หยุดหาก 3 Epoch ไม่มี improvement
)


C:\Users\narue\AppData\Local\Temp\ipykernel_18976\3293059516.py:29: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [20]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision Neg,Recall Neg,F1 Neg,Support Neg,Precision Pos,Recall Pos,F1 Pos,Support Pos
1,0.672400,0.652851,0.684864,0.707865,0.626866,0.664908,201,0.666667,0.742574,0.702576,202
2,0.537400,0.678476,0.528536,0.518771,0.756219,0.615385,201,0.554545,0.301980,0.391026,202
3,0.303400,0.491446,0.724566,0.685950,0.825871,0.749436,201,0.782609,0.623762,0.694215,202
4,0.139500,0.302849,0.940447,0.973262,0.905473,0.938144,201,0.912037,0.975248,0.942584,202
5,0.074500,0.138239,0.965261,0.984456,0.945274,0.964467,201,0.947619,0.985149,0.966019,202
6,0.052300,0.066575,0.990074,1.000000,0.980100,0.989950,201,0.980583,1.000000,0.990196,202
7,0.020900,0.102862,0.985112,1.000000,0.970149,0.984848,201,0.971154,1.000000,0.985366,202
8,0.007200,0.052413,0.990074,0.994975,0.985075,0.990000,201,0.985294,0.995050,0.990148,202
9,0.026100,0.020169,0.995037,1.000000,0.990050,0.995000,201,0.990196,1.000000,0.995074,202
10,0.009000,0.017800,0.995037,1.000000,0.990050,0.995000,201,0.990196,1.000000,0.995074,202


c:\Users\narue\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\narue\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\narue\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mod

TrainOutput(global_step=1212, training_loss=0.1715178572378358, metrics={'train_runtime': 459.8939, 'train_samples_per_second': 42.062, 'train_steps_per_second': 2.635, 'total_flos': 1272405063720960.0, 'train_loss': 0.1715178572378358, 'epoch': 12.0})

In [21]:
eval_result = trainer.evaluate()
print("Evaluation result:", eval_result)
#trainer.save_model("my_wangchanberta_sentiment_model_cutall")
trainer.save_model("my_wangchanberta_sentiment_model")
# trainer.save_model("my_wangchanberta_sentiment_model_cutall_double")

c:\Users\narue\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\narue\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\narue\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mod

Evaluation result: {'eval_loss': 0.02016945369541645, 'eval_accuracy': 0.9950372208436724, 'eval_precision_neg': 1.0, 'eval_recall_neg': 0.9900497512437811, 'eval_f1_neg': 0.995, 'eval_support_neg': 201, 'eval_precision_pos': 0.9901960784313726, 'eval_recall_pos': 1.0, 'eval_f1_pos': 0.9950738916256158, 'eval_support_pos': 202, 'eval_runtime': 2.737, 'eval_samples_per_second': 147.243, 'eval_steps_per_second': 18.634, 'epoch': 12.0}


In [22]:
predictions_output = trainer.predict(tokenized_datasets["test"])
pred_logits = predictions_output.predictions
pred_labels = np.argmax(pred_logits, axis=-1)
true_labels = predictions_output.label_ids

# id2label = {0: "neg", 1: "neu", 2: "pos"}
id2label = {0: "neg", 1: "pos"}

result_df = pd.DataFrame({
    "text": test_df["paragraph"].tolist(),
    "true_label": [id2label[l] for l in true_labels],
    "predicted_label": [id2label[l] for l in pred_labels]
})

print(result_df.head(10))

result_df.to_csv("prediction_results_2.csv", index=False, encoding="utf-8-sig")

cm = confusion_matrix(true_labels, pred_labels)
print("Confusion Matrix:\n", cm)

c:\Users\narue\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\narue\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\narue\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mod

                                                text true_label  \
0    เทคโนโลยีการเงินมีศักยภาพดีความสำเร็จไม่ยั่งยืน        pos   
1                        น้ำมันดิบความกดดันสร้างพลัง        neg   
2              กองทุนรวมเคลื่อนไหวดีการวางแผนล้มเหลว        pos   
3  ดอกเบี้ยพันธบัตรรัฐบาลขยับขึ้นดีเสถียรภาพสั่นค...        pos   
4    การลงทุนตราสารอนุพันธ์ความเสียหายฟื้นฟูได้ลำบาก        neg   
5      ราคาน้ำมันดิบอิหร่านเติบโตแข็งแกร่งคุณภาพตกลง        pos   
6                   ตราสารหนี้ความลำบากคือความสำเร็จ        neg   
7  การลงทุนในกองทุนรวมความเจ็บปวดคือหนทางสู่การรั...        neg   
8   ราคาขายปลีกน้ำมันความลำบากคือหนทางสู่ความแกร่ง          neg   
9             อัตราดอกเบี้ยเงินฝากพุ่งแรงจากการแก้ไข        pos   

  predicted_label  
0             pos  
1             neg  
2             pos  
3             pos  
4             neg  
5             pos  
6             neg  
7             neg  
8             neg  
9             pos  
Confusion Matrix:
 [[199   2]
 [  0 202]]


In [23]:
wrong_preds = result_df[result_df["true_label"] != result_df["predicted_label"]]
print("mistake prediction example:\n", wrong_preds.head(10))

correct_preds = result_df[result_df["true_label"] == result_df["predicted_label"]]
print("correctness prediction example:\n", correct_preds.head(10))

mistake prediction example:
                                         text true_label predicted_label
212          หุ้นไทยความท้าทายสร้างการเติบโต        neg             pos
252  การลงทุนเพื่อลดหย่อนภาษีตกต่ำฟื้นตัวช้า        neg             pos
correctness prediction example:
                                                 text true_label  \
0    เทคโนโลยีการเงินมีศักยภาพดีความสำเร็จไม่ยั่งยืน        pos   
1                        น้ำมันดิบความกดดันสร้างพลัง        neg   
2              กองทุนรวมเคลื่อนไหวดีการวางแผนล้มเหลว        pos   
3  ดอกเบี้ยพันธบัตรรัฐบาลขยับขึ้นดีเสถียรภาพสั่นค...        pos   
4    การลงทุนตราสารอนุพันธ์ความเสียหายฟื้นฟูได้ลำบาก        neg   
5      ราคาน้ำมันดิบอิหร่านเติบโตแข็งแกร่งคุณภาพตกลง        pos   
6                   ตราสารหนี้ความลำบากคือความสำเร็จ        neg   
7  การลงทุนในกองทุนรวมความเจ็บปวดคือหนทางสู่การรั...        neg   
8   ราคาขายปลีกน้ำมันความลำบากคือหนทางสู่ความแกร่ง          neg   
9             อัตราดอกเบี้ยเงินฝากพุ่งแรงจากการแก้ไข

In [24]:
# Evaluate -> จะได้ acc, f1, precision, recall
eval_result = trainer.evaluate()
print("Evaluation metrics:", eval_result)

# Confusion Matrix
pred_output = trainer.predict(tokenized_datasets["test"])
pred_labels = np.argmax(pred_output.predictions, axis=-1)
true_labels = pred_output.label_ids
print("pred_output:\n", pred_output)
print("pred_labels:\n", pred_labels)
print("true_labels:\n", true_labels)

cm = confusion_matrix(true_labels, pred_labels)
print("Confusion Matrix (raw):\n", cm)

# label_names = ["neg", "pos"]  # ถ้ามี 3 label
label_names = ["neg", "pos", "neu"]  # ถ้ามี 3 label
df_cm = pd.DataFrame(cm, index=label_names, columns=label_names)
print("Confusion Matrix (DataFrame):\n", df_cm)

Evaluation metrics: {'eval_loss': 0.02016945369541645, 'eval_accuracy': 0.9950372208436724, 'eval_precision_neg': 1.0, 'eval_recall_neg': 0.9900497512437811, 'eval_f1_neg': 0.995, 'eval_support_neg': 201, 'eval_precision_pos': 0.9901960784313726, 'eval_recall_pos': 1.0, 'eval_f1_pos': 0.9950738916256158, 'eval_support_pos': 202, 'eval_runtime': 2.9746, 'eval_samples_per_second': 135.482, 'eval_steps_per_second': 17.145, 'epoch': 12.0}


c:\Users\narue\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\narue\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\narue\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mod

pred_output:
 PredictionOutput(predictions=array([[-5.3317394 ,  4.650378  ],
       [ 5.3192563 , -4.5354185 ],
       [-5.3473067 ,  4.772648  ],
       [-4.0381026 ,  3.2944477 ],
       [ 5.462409  , -4.753555  ],
       [-3.455393  ,  3.387218  ],
       [ 5.5120196 , -4.7014604 ],
       [ 5.501211  , -4.7345643 ],
       [ 5.270206  , -4.6129127 ],
       [-4.3168406 ,  3.6869314 ],
       [-3.4492753 ,  3.3831992 ],
       [-5.3633547 ,  4.726311  ],
       [ 5.544798  , -4.7698903 ],
       [ 5.3721957 , -4.695521  ],
       [ 5.321566  , -4.686264  ],
       [-3.472583  ,  3.3983126 ],
       [-5.2986197 ,  4.580377  ],
       [-5.2805176 ,  4.6063714 ],
       [ 5.5734687 , -4.7472677 ],
       [-5.36612   ,  4.6228957 ],
       [ 5.0661583 , -4.3715525 ],
       [-3.4711568 ,  3.3979063 ],
       [ 5.360458  , -4.6018777 ],
       [ 5.1878376 , -4.4748087 ],
       [ 5.5587554 , -4.714344  ],
       [ 5.2525234 , -4.4487953 ],
       [-3.459081  ,  3.3900385 ],
       [-3.4

ValueError: Shape of passed values is (2, 2), indices imply (3, 3)

In [ ]:
from itertools import cycle
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import PrecisionRecallDisplay, precision_recall_curve, average_precision_score

# ข้อมูลที่มีอยู่จริง
print("Shape of predictions:", pred_output.predictions.shape)  # (6, 2)
print("Unique values in true_labels:", np.unique(true_labels))  # [0 1]

# Mapping labels
label2id = {"neg": 0, "pos": 1}
id2label = {v: k for k, v in label2id.items()}

# ดึงค่า F1-score ที่คำนวณได้จากโมเดล
eval_result = trainer.evaluate()
f1_score_avg = eval_result["eval_f1"]

# กำหนดค่า iso-F1 scores แบบไดนามิก
f_scores = np.linspace(f1_score_avg * 0.8, f1_score_avg * 1.2, num=4)
print("Using dynamic F1-scores:", f_scores)

# คำนวณ Precision-Recall Curve
precision = {}
recall = {}
average_precision = {}

# สำหรับคลาส neg (0)
precision[0], recall[0], _ = precision_recall_curve(true_labels == 0, pred_output.predictions[:, 0])
average_precision[0] = average_precision_score(true_labels == 0, pred_output.predictions[:, 0])

# สำหรับคลาส pos (1)
precision[1], recall[1], _ = precision_recall_curve(true_labels == 1, pred_output.predictions[:, 1])
average_precision[1] = average_precision_score(true_labels == 1, pred_output.predictions[:, 1])

# Micro-average - สร้าง one-hot encoding ด้วยตัวเอง
y_true_binary = np.zeros((len(true_labels), 2))
for i, label in enumerate(true_labels):
    y_true_binary[i, label] = 1

precision["micro"], recall["micro"], _ = precision_recall_curve(
    y_true_binary.ravel(), pred_output.predictions.ravel()
)
average_precision["micro"] = average_precision_score(
    y_true_binary, pred_output.predictions, average="micro"
)

# ตั้งค่าสีสำหรับกราฟ
colors = ["navy", "darkorange", "gold"]

_, ax = plt.subplots(figsize=(7, 8))

# วาดเส้น iso-F1 curves
for f_score in f_scores:
    x = np.linspace(0.01, 1)
    y = f_score * x / (2 * x - f_score)
    plt.plot(x[y >= 0], y[y >= 0], color="gray", alpha=0.2)
    plt.annotate(f"f1={f_score:0.2f}", xy=(0.9, y[45] + 0.02))

# วาด Micro-average Precision-Recall Curve
display = PrecisionRecallDisplay(
    recall=recall["micro"],
    precision=precision["micro"]
)
display.plot(ax=ax, name="Micro-average precision-recall", color=colors[2])

# วาด Precision-Recall Curve สำหรับแต่ละคลาส
for i in range(2):  # มี 2 คลาส
    display = PrecisionRecallDisplay(
        recall=recall[i],
        precision=precision[i]
    )
    display.plot(ax=ax, name=f"Precision-Recall for {id2label[i]}", color=colors[i])

# ตั้งค่า Legend และชื่อกราฟ
ax.legend(loc="best")
ax.set_title("Precision-Recall Curve for Binary Classification")

plt.show()

Shape of predictions: (860, 2)
Unique values in true_labels: [0 1]


KeyError: 'eval_f1'

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import pandas as pd
import numpy as np

# 1) Load Model and tokenizer (using model path which already fine-tune set up)
model_path = "my_wangchanberta_sentiment_model"  # or other names
model = AutoModelForSequenceClassification.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

# Move to GPU if you have
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# Create new unknown paragraph
new_texts = [
    "วันนี้ข่าวดีมาก น่าลงทุน",
    "แย่มาก ไม่พอใจการทิศทางของกราฟเลย",
    "ก็โอเคนะ กราฟหุ้นตัวนี้ ไม่ได้แย่แต่ก็ไม่ดีมาก",
    "ชอบบรรยากาศในหุ้นตัวนี้มากมาก",
    "หุ้นตก ทิ้งดิ่ง รู้สึกแย่",
    "หุ้นตัวนี้เฉย ๆ ไม่ประทับใจเท่าไร",
    "หุ้นตัวนี้ ผลประกอบการดีเยี่ยม",
    "รู้สึกงงกับระบบการจองหุ้นตัวนี้",
    "หุ้นตัวสุดยอดเลย! ต้องซื้อเพิ่มอีกแน่นอน",
    "SJWD แย้มไตรมาส 4 สดใส มั่นใจดันรายได้ปีนี้แตะ 2.5 หมื่นล้าน",
    "เม็ดเงิน วายุภักษ์ ซื้อหุ้นค้าปลีก ดัน SET ปิดบวก 8 จุด แนวโน้มพรุ่งไซด์เวย์อัพ",
    "SABUY อ่วมหนัก Q3 ขาดทุน 865 ล้านบาท ตลท. แขวนป้าย CB มีผล 18 พ.ย.นี้",
    "เช้าวันนี้SETHD ปิดที่ระดับ 1,075.29  จุด ลดลง 7.48  จุด หรือ 0.69 %",
    "ต่างชาติ เทขาย 3.4 พันล้านบาท กด SET ปิดลบ 9 จุด"
]

# unknown paragraph Tokenize 
encoding = tokenizer(
    new_texts,
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

# Move to GPU if you have
encoding = {k: v.to(device) for k, v in encoding.items()}

# unknown paragraph prediction
with torch.no_grad():
    outputs = model(**encoding)
    logits = outputs.logits
    preds = torch.argmax(logits, dim=-1)

# convert label back to text symbol (neg, neu, pos)
#id2label = {0: "neg", 1: "pos"}
id2label = {0: "neg", 1: "neu", 2: "pos"}
predicted_labels = [id2label[p.item()] for p in preds]

# Create dataframe
df_results = pd.DataFrame({
    "text": new_texts,
    "predicted_label": predicted_labels
})

print(df_results)

df_results.to_csv("inference_results_2.csv", index=False, encoding="utf-8-sig")

                                                 text predicted_label
0                            วันนี้ข่าวดีมาก น่าลงทุน             pos
1                   แย่มาก ไม่พอใจการทิศทางของกราฟเลย             neg
2      ก็โอเคนะ กราฟหุ้นตัวนี้ ไม่ได้แย่แต่ก็ไม่ดีมาก             neg
3                       ชอบบรรยากาศในหุ้นตัวนี้มากมาก             neg
4                           หุ้นตก ทิ้งดิ่ง รู้สึกแย่             neg
5                   หุ้นตัวนี้เฉย ๆ ไม่ประทับใจเท่าไร             neg
6                      หุ้นตัวนี้ ผลประกอบการดีเยี่ยม             pos
7                     รู้สึกงงกับระบบการจองหุ้นตัวนี้             neg
8            หุ้นตัวสุดยอดเลย! ต้องซื้อเพิ่มอีกแน่นอน             neg
9   SJWD แย้มไตรมาส 4 สดใส มั่นใจดันรายได้ปีนี้แตะ...             neg
10  เม็ดเงิน วายุภักษ์ ซื้อหุ้นค้าปลีก ดัน SET ปิด...             neg
11  SABUY อ่วมหนัก Q3 ขาดทุน 865 ล้านบาท ตลท. แขวน...             neg
12  เช้าวันนี้SETHD ปิดที่ระดับ 1,075.29  จุด ลดลง...             neg
13   ต่างชาติ เทขาย 